In [ ]:
from tqdm import tqdm
import pyarrow.ipc as ipc

with tqdm(total=n_rows, unit_scale=True, desc='Reading GTEx TSV') as pbar:
    for chunk in csv_reader:
        pbar.update(len(chunk))
        if writer is None:
            writer = ipc.RecordBatchFileWriter(sink, chunk.schema)
        writer.write_batch(chunk)

csv_reader.close()
if writer is not None:
    writer.close()
sink.close()

In [ ]:
with pa.memory_map(arrow_file, 'r') as source:
    reader = ipc.RecordBatchFileReader(source)
    gtex_table = reader.read_all()
print(f'Finished reading {arrow_file} with {gtex_table.num_rows} rows and {gtex_table.num_columns} columns')

gtex_df = gtex_table.to_pandas(self_destruct=True)
print("Finished converting to pandas DataFrame:", gtex_df.shape)
gtex_df.to_pickle('gtex_junctions_df.pkl')
print("Saved GTEx junctions DataFrame to gtex_junctions_df.pkl")

# concatenate all chunks into a single pyarrow table
# gtex_table = pa.Table.from_batches(batch_list)
# gtex_df = gtex_table.to_pandas()
# print(gtex_df.shape)
# max_value = -1
# for chunk in table_chunks:
#     chunk_df = chunk.to_pandas()
#     # find the max value in chunk_df (excluding the first two columns)
#     max_value_in_chunk = chunk_df.iloc[:, 2:].max().max()
#     if max_value < max_value_in_chunk:
#         max_value = max_value_in_chunk
#         print(f"New max value: {max_value}")


In [ ]:
import pandas as pd
gtex_df = pd.read_pickle('data/gtex_v10/gtex_junctions_df.pkl')
print(gtex_df.shape)
print(gtex_df.columns[:7])
print(gtex_df.iloc[:3, :3])

In [ ]:
# save gtex_df to csv without the index
gtex_df.to_csv('data/gtex_v10/gtex_junctions_df.csv', index=False)
print("Saved GTEx junctions DataFrame to gtex_junctions_df.csv")

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
all_values = gtex_df.iloc[:, 2:].values.flatten()
plt.figure(figsize=(10,6))
plt.hist(all_values, bins=100, color='blue', alpha=0.7, range=(0.001, np.percentile(all_values, 99)))
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.title('Histogram of GTEx Junctions Values')
plt.show()